# Projeto Análise da influência do saneamento básico no pico de letalidade de doenças DRSAI

Pergunta 1 — Classificação regional com saneamento como feature central:
"Dado o perfil de saneamento básico (cobertura de água tratada, esgoto, coleta de lixo) e o cenário epidemiológico inicial (casos notificados nas últimas N semanas) de um município brasileiro, qual o risco preditivo (Alto/Médio/Baixo) de ele enfrentar um pico de letalidade hospitalar por doenças infecciosas de transmissão hídrica/fecal-oral nos próximos M meses?"

Pergunta 2 — Foco em doenças específicas de transmissão hídrica:
"Para doenças infecciosas reconhecidamente associadas à qualidade do saneamento (leptospirose, hepatite A, esquistossomose, doenças diarreicas agudas, febre tifoide), qual o risco de pico de letalidade hospitalar em uma região, dadas as condições sanitárias e o cenário epidemiológico inicial?"

- Risco preditivo de pico de letalidade

### Unidade analítica 

- Macrorregião de saúde por residência x trimestre
- DRSAI de doenças fecal-oral

### Escopo do projeto

|Item | Decisão|
|-------|-------|
|Fonte primária do target|SIH-SUS (internações + óbitos hospitalares)|
|Fonte complementar|SINAN (casos notificados como feature antecedente)|
|Critério de inclusão de doenças|CID-10 nos capítulos DRSAI oficiais|
|Capítulos CID-10 principais|A00-A09, A27, B15, B65-B83|
|Postura de modelagem|Agregada (categoria DRSAI como feature)|
|Fonte primária do target|SIH-SUS (internações + óbitos hospitalares)|


In [21]:
import sys
print(sys.executable)

/home/victoria/Documentos/Tópicos de banco de dados/projeto-datasus-drsai/venv/bin/python


In [22]:
from pysus.ftp.databases.sih import SIH
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

In [23]:
sih = SIH().load()
print(type(sih))

<class 'pysus.ftp.databases.sih.SIH'>


In [24]:
arquivos = sih.get_files(group="RD", uf="SP", year=2024, month=1)
print(f"Tipo do retorno: {type(arquivos)}")
print(f"Quantidade: {len(arquivos)}")
print(f"Primeiro arquivo: {arquivos[0]}")

Tipo do retorno: <class 'list'>
Quantidade: 1
Primeiro arquivo: RDSP2401.dbc


In [25]:
arquivo_teste = arquivos[0]
print(f"Nome: {arquivo_teste}")
print(f"Atributos públicos: {[a for a in dir(arquivo_teste) if not a.startswith('_')]}")

Nome: RDSP2401.dbc
Atributos públicos: ['async_download', 'basename', 'download', 'extension', 'info', 'name', 'parent_path', 'path']


In [26]:
# info - ver detalhes do arquivo antes de baixar
print(f"Nome: {arquivo_teste.name}")
print(f"Basename: {arquivo_teste.basename}")
print(f"Extensão: {arquivo_teste.extension}")
print(f"Path no FTP: {arquivo_teste.path}")
print(f"Parent path: {arquivo_teste.parent_path}")
print(f"Info: {arquivo_teste.info}")

Nome: RDSP2401
Basename: RDSP2401.dbc
Extensão: .dbc
Path no FTP: /dissemin/publicos/SIHSUS/200801_/Dados/RDSP2401.dbc
Parent path: /dissemin/publicos/SIHSUS/200801_/Dados
Info: {'size': '17.5 MB', 'type': 'DBC file', 'modify': '2025-02-09 09:29PM'}


In [27]:
# verificar a pasta
pasta_raw_sih = Path("../data/raw/sih")
pasta_raw_sih.mkdir(parents=True, exist_ok=True)
print(f"Pasta criada em: {pasta_raw_sih.resolve()}")

Pasta criada em: /home/victoria/Documentos/Tópicos de banco de dados/projeto-datasus-drsai/data/raw/sih


In [28]:
# download de teste 
resultado = sih.download(arquivos, local_dir=str(pasta_raw_sih))
print(f"Tipo do resultado: {type(resultado)}")
print(f"Resultado: {resultado}")

17462541it [00:00, 7126905280.38it/s]

Tipo do resultado: <class 'pysus.data.local.ParquetSet'>
Resultado: ../data/raw/sih/RDSP2401.parquet


In [29]:
arquivos_baixados = list(pasta_raw_sih.iterdir())
for item in arquivos_baixados:
    tipo = "pasta" if item.is_dir() else "arquivo"
    print(f"[{tipo}] {item.name}")

[pasta] RDSP2401.parquet
[pasta] RDCE2401.parquet


In [30]:
# Abrir o Parquet (funciona tanto para arquivo único quanto para diretório)
caminho_arquivo = pasta_raw_sih / "RDSP2401.parquet"

# Primeiro, ver o schema sem carregar tudo na memória
dataset = pq.ParquetDataset(str(caminho_arquivo))
schema = dataset.schema
print(f"Número de colunas: {len(schema.names)}")
print(f"\nPrimeiras 10 colunas com tipos:")
for i, nome in enumerate(schema.names[:10]):
    tipo = schema.field(nome).type
    print(f"  {i+1:3d}. {nome:20s} → {tipo}")

Número de colunas: 113

Primeiras 10 colunas com tipos:
    1. UF_ZI                → string
    2. ANO_CMPT             → string
    3. MES_CMPT             → string
    4. ESPEC                → string
    5. CGC_HOSP             → string
    6. N_AIH                → string
    7. IDENT                → string
    8. CEP                  → string
    9. MUNIC_RES            → string
   10. NASC                 → string


In [31]:
todas_colunas = schema.names  # já é uma lista
print(f"Total de colunas: {len(todas_colunas)}")
print(f"\nLista completa:")
for i, nome in enumerate(todas_colunas):
    print(f"  {i+1:3d}. {nome}")

Total de colunas: 113

Lista completa:
    1. UF_ZI
    2. ANO_CMPT
    3. MES_CMPT
    4. ESPEC
    5. CGC_HOSP
    6. N_AIH
    7. IDENT
    8. CEP
    9. MUNIC_RES
   10. NASC
   11. SEXO
   12. UTI_MES_IN
   13. UTI_MES_AN
   14. UTI_MES_AL
   15. UTI_MES_TO
   16. MARCA_UTI
   17. UTI_INT_IN
   18. UTI_INT_AN
   19. UTI_INT_AL
   20. UTI_INT_TO
   21. DIAR_ACOM
   22. QT_DIARIAS
   23. PROC_SOLIC
   24. PROC_REA
   25. VAL_SH
   26. VAL_SP
   27. VAL_SADT
   28. VAL_RN
   29. VAL_ACOMP
   30. VAL_ORTP
   31. VAL_SANGUE
   32. VAL_SADTSR
   33. VAL_TRANSP
   34. VAL_OBSANG
   35. VAL_PED1AC
   36. VAL_TOT
   37. VAL_UTI
   38. US_TOT
   39. DT_INTER
   40. DT_SAIDA
   41. DIAG_PRINC
   42. DIAG_SECUN
   43. COBRANCA
   44. NATUREZA
   45. NAT_JUR
   46. GESTAO
   47. RUBRICA
   48. IND_VDRL
   49. MUNIC_MOV
   50. COD_IDADE
   51. IDADE
   52. DIAS_PERM
   53. MORTE
   54. NACIONAL
   55. NUM_PROC
   56. CAR_INT
   57. TOT_PT_SP
   58. CPF_AUT
   59. HOMONIMO
   60. NUM_FILHOS
   6

In [32]:
tabela = pq.read_table(str(caminho_arquivo))
df = tabela.to_pandas()
print(f"Shape: {df.shape}")  # (linhas, colunas)
print(f"Memória usada: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Shape: (222849, 113)
Memória usada: 1325.7 MB


## Exemplo de extração e análise com SP para Reduzidas - ano 2024 e mês de Janeiro (Doenças gerais)

In [33]:
# Inspecionar as colunas-chave para o projeto
colunas_chave = ['DIAG_PRINC', 'MUNIC_RES', 'MORTE', 'DT_INTER', 'DT_SAIDA', 
                 'IDADE', 'SEXO', 'DIAS_PERM']

colunas_presentes = [c for c in colunas_chave if c in df.columns]
colunas_faltantes = [c for c in colunas_chave if c not in df.columns]

print(f"Colunas presentes ({len(colunas_presentes)}): {colunas_presentes}")
if colunas_faltantes:
    print(f"Colunas faltantes ({len(colunas_faltantes)}): {colunas_faltantes}")

print(f"\nPrimeiras 5 linhas das colunas-chave:")
print(df[colunas_presentes].head())

Colunas presentes (8): ['DIAG_PRINC', 'MUNIC_RES', 'MORTE', 'DT_INTER', 'DT_SAIDA', 'IDADE', 'SEXO', 'DIAS_PERM']

Primeiras 5 linhas das colunas-chave:
  DIAG_PRINC MUNIC_RES MORTE  DT_INTER  DT_SAIDA IDADE SEXO DIAS_PERM
0       K800    353060     0  20240123  20240125    57    3         2
1       N47     353060     0  20240128  20240129    14    1         1
2       N47     353060     0  20240116  20240117    51    1         1
3       N508    353060     0  20240116  20240117     3    1         1
4       Z302    353060     0  20240128  20240130    31    3         2


### Colunas utilizadas pra análise no projeto

DIAG_PRINC — CID-10 do diagnóstico principal \
DIAG_SECUN — CID-10 secundário \
MUNIC_RES — Município de RESIDÊNCIA do paciente \
MUNIC_MOV — Município do hospital \
MORTE — Se o paciente morreu (S/N ou 0/1) \
DT_INTER — Data de internação \
DT_SAIDA — Data de saída\
IDADE / COD_IDADE — Idade (tem código que indica se está em dias/meses/anos)\
SEXO — Sexo\
DIAS_PERM — Dias de permanência\
VAL_TOT — Valor total da internação\
UTI_MES_TO — Dias em UTI no mês

**Diagnósticos secundários extendidos (colunas 96-113)**\
DIAGSEC1 até DIAGSEC9 + TPDISEC1 até TPDISEC9. Significa que cada internação pode ter até 9 diagnósticos secundários além do principal. 
Exemplo: Um paciente pode ser internado por pneumonia (DIAG_PRINC) mas ter leptospirose como DIAG_SECUN — e leptospirose é DRSAI

**Marcadores de gravidade**
- MARCA_UTI - Ida para UTI
- UTI_MES_TO - dias em UTI
- CID_MORTE - CID-10 da causa do óbito


#### Para expansão posterior
**valores financeiros (colunas 25-37, 90-94)**
- VAL_SH, VAL_SP, VAL_TOT, VAL_UTI, VAL_UCI (a considerar posteriormente)


**Informações sócio-demográficas** 
- RACA_COR, ETNIA, INSTRU (grau de instrução), NASC (data de nascimento). Úteis para análises estratificadas.

|**OBSERVAÇÃO**|
|--------------|
|Todos os dados são strings. Vamos precisar converter tudo na fase de ETL|

**Exemplo com SP ano 2024 e mês 1**\
DIAG_PRINC: K800      (string de 4 caracteres)\
MUNIC_RES:  353060    (string, código IBGE de 6 dígitos)\
MORTE:      0         (string "0" ou "1", NÃO booleano)\
DT_INTER:   20240123  (string no formato YYYYMMDD)\
DT_SAIDA:   20240125  (string no formato YYYYMMDD)\
IDADE:      57        (string --> Ver COD_IDADE)\
SEXO:       3         (string! 1=Masc, 3=Fem — não é 'M'/'F')\
DIAS_PERM:  2         (string numérica)

**Sobre COD_IDADE**
Códigos possíveis de COD_IDADE:\
2 = dias\
3 = meses\
4 = anos\
5 = mais de 100 anos\
--> É preciso converter tudo pra idade em anos para não causar confusão

**Sobre DIAG_PRINC**\
DIAG_PRINC = K800 == K80.0 no CID-10 → cálculo biliar.


## Montando a lista dos CIDs do DRSAI
- A00-A09 — Doenças infecciosas intestinais
- A27 — Leptospirose
- B15 — Hepatite A aguda
- B65-B83 — Helmintíases

Em dados reais do CID-10, cada categoria se desdobra em códigos de 4 caracteres. Por exemplo, A00 vira A000 (cólera por Vibrio cholerae 01), A001 (cólera por V. cholerae El Tor), A009 (cólera não especificada), etc.

In [34]:
# prefixos do DRSAI CID-10

cids_drsai = {
    'A00': 'Cólera',
    'A01': 'Febres tifoide e paratifoide',
    'A02': 'Outras infecções por Salmonella',
    'A03': 'Shiguelose',
    'A04': 'Outras infecções bacterianas intestinais',
    'A05': 'Outras intoxicações alimentares bacterianas',
    'A06': 'Amebíase',
    'A07': 'Outras doenças intestinais por protozoários',
    'A08': 'Infecções intestinais virais',
    'A09': 'Diarreia e gastroenterite origem infecciosa presumível',
    'A27': 'Leptospirose',
    'B15': 'Hepatite A aguda',
    'B65': 'Esquistossomose',
    'B66': 'Outras infestações por trematódeos',
    'B67': 'Equinococose',
    'B68': 'Infestação por Taenia',
    'B69': 'Cisticercose',
    'B70': 'Difilobotríase e esparganose',
    'B71': 'Outras infestações por cestoides',
    'B72': 'Dracunculíase',
    'B73': 'Oncocercose',
    'B74': 'Filariose',
    'B75': 'Triquinose',
    'B76': 'Ancilostomíase',
    'B77': 'Ascaridíase',
    'B78': 'Estrongiloidíase',
    'B79': 'Tricuríase',
    'B80': 'Enterobíase',
    'B81': 'Outras helmintíases intestinais',
    'B82': 'Parasitoses intestinais não especificadas',
    'B83': 'Outras helmintíases',
}

prefixos_drsai = list(cids_drsai.keys())
print(f"Total de prefixos CID DRSAI: {len(prefixos_drsai)}")
print(f"Prefixos: {prefixos_drsai}")

Total de prefixos CID DRSAI: 31
Prefixos: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A27', 'B15', 'B65', 'B66', 'B67', 'B68', 'B69', 'B70', 'B71', 'B72', 'B73', 'B74', 'B75', 'B76', 'B77', 'B78', 'B79', 'B80', 'B81', 'B82', 'B83']


In [35]:
# filtro do DIAG_PRINC começando com prefixo DRSAI
# TRUE para linhas do DIAG_PRINC que começa com algum dos prefixos

mascara_principal = df['DIAG_PRINC'].str.startswith(tuple(prefixos_drsai), na=False)
df_drsai_principal = df[mascara_principal].copy()

print(f"Total de internações em SP-jan/2024: {len(df):,}")
print(f"Internações DRSAI por DIAG_PRINC: {len(df_drsai_principal):,}")
print(f"Percentual: {len(df_drsai_principal) / len(df) * 100:.2f}%")

Total de internações em SP-jan/2024: 222,849
Internações DRSAI por DIAG_PRINC: 1,030
Percentual: 0.46%


In [36]:
# distribuição por prefixo
df_drsai_principal['PREFIXO_CID'] = df_drsai_principal['DIAG_PRINC'].str[:3]

distribuicao = df_drsai_principal['PREFIXO_CID'].value_counts()
print("Distribuição de internações DRSAI por capítulo CID:\n")
for prefixo, count in distribuicao.items():
    nome = cids_drsai.get(prefixo, '(prefixo não mapeado)')
    print(f"  {prefixo}  {nome:50s} → {count:,}")

Distribuição de internações DRSAI por capítulo CID:

  A09  Diarreia e gastroenterite origem infecciosa presumível → 782
  A08  Infecções intestinais virais                       → 81
  A04  Outras infecções bacterianas intestinais           → 78
  A27  Leptospirose                                       → 25
  B15  Hepatite A aguda                                   → 13
  A06  Amebíase                                           → 9
  A05  Outras intoxicações alimentares bacterianas        → 8
  A07  Outras doenças intestinais por protozoários        → 6
  A02  Outras infecções por Salmonella                    → 6
  B82  Parasitoses intestinais não especificadas          → 5
  A00  Cólera                                             → 4
  B69  Cisticercose                                       → 3
  B65  Esquistossomose                                    → 3
  B77  Ascaridíase                                        → 2
  B83  Outras helmintíases                                → 2
  B81  

A letalidade hospitalar vai ser usada para prever os resultados. Se a letalidade hospitalar de SP é 2.23%, se uma macrorregião tiver letalidade de 15% é bem maior que o normal

In [37]:
# letalidade hospitalar rela

print(f"Valores únicos em MORTE: {df_drsai_principal['MORTE'].unique()}")
print(f"Tipo de dado: {df_drsai_principal['MORTE'].dtype}")

# converte morte pra int
df_drsai_principal['MORTE_INT'] = df_drsai_principal['MORTE'].astype(int)

total_internacoes = len(df_drsai_principal)
total_obitos = df_drsai_principal['MORTE_INT'].sum()
letalidade = total_obitos / total_internacoes * 100

print(f"\n--- Letalidade hospitalar DRSAI em SP-jan/2024 ---")
print(f"Total internações DRSAI: {total_internacoes:,}")
print(f"Óbitos hospitalares DRSAI: {total_obitos:,}")
print(f"Letalidade: {letalidade:.2f}%")

Valores únicos em MORTE: ['0' '1']
Tipo de dado: object

--- Letalidade hospitalar DRSAI em SP-jan/2024 ---
Total internações DRSAI: 1,030
Óbitos hospitalares DRSAI: 23
Letalidade: 2.23%


In [38]:
# letalidade por capítulo
letalidade_por_cid = df_drsai_principal.groupby('PREFIXO_CID').agg(
    internacoes=('N_AIH', 'count'),
    obitos=('MORTE_INT', 'sum')
).reset_index()

letalidade_por_cid['letalidade_pct'] = (
    letalidade_por_cid['obitos'] / letalidade_por_cid['internacoes'] * 100
).round(2)

letalidade_por_cid['nome'] = letalidade_por_cid['PREFIXO_CID'].map(cids_drsai)

# ordena por letalidade decrescente
letalidade_por_cid = letalidade_por_cid.sort_values('letalidade_pct', ascending=False)

print("Letalidade hospitalar por capítulo DRSAI em SP-jan/2024:\n")
print(letalidade_por_cid[['PREFIXO_CID', 'nome', 'internacoes', 'obitos', 'letalidade_pct']].to_string(index=False))

Letalidade hospitalar por capítulo DRSAI em SP-jan/2024:

PREFIXO_CID                                                   nome  internacoes  obitos  letalidade_pct
        B83                                    Outras helmintíases            2       1           50.00
        A06                                               Amebíase            9       1           11.11
        A27                                           Leptospirose           25       1            4.00
        A08                           Infecções intestinais virais           81       3            3.70
        A04               Outras infecções bacterianas intestinais           78       2            2.56
        A09 Diarreia e gastroenterite origem infecciosa presumível          782      15            1.92
        A01                           Febres tifoide e paratifoide            1       0            0.00
        A00                                                 Cólera            4       0            0.00
      

### Conclusões do teste em SP:

1. Distribuição muito concentrada para A09 Diarreia e gastroenterite origem infecciosa presumível 
- Os 5 CIDs principais (A09, A08, A04, A27, B15) somam 95% dos casos
- Os outros 25 CIDs somam 5% juntos
--> mas é preciso ver em outros lugares também, pois SP possui saneamento básico relativamente muito bom e é só 1 mês em 1 ano

2. Talvez seja melhor agrupar CIDs por categorias
- infecções fecal-orais = A00-A09+B15
- helmintíases" = B65-B83
- leptospirose = A27
- o resto fica como "outras" (???)

3. Não dá pra fazer letalidade com CID individuais
- Ex.: Outras helmintíases            2       1           50.00 (muito inflado)

4. É melhor observar como o DIAG_SECUN se comporta. Se vai trazer mais casos ou se vai ser só overkill

LEMBRAR DE FAZER COM O DO NORDESTE PRA VER SE TEM DADOS MELHORES (PIORES NO CASO)

## Teste com o estado do ceará

In [39]:
arquivos = sih.get_files(group="RD", uf="CE", year=2024, month=1)
print(f"Tipo do retorno: {type(arquivos)}")
print(f"Quantidade: {len(arquivos)}")
print(f"Primeiro arquivo: {arquivos[0]}")

Tipo do retorno: <class 'list'>
Quantidade: 1
Primeiro arquivo: RDCE2401.dbc


In [40]:
arquivo_teste = arquivos[0]
print(f"Nome: {arquivo_teste}")
print(f"Atributos públicos: {[a for a in dir(arquivo_teste) if not a.startswith('_')]}")

Nome: RDCE2401.dbc
Atributos públicos: ['async_download', 'basename', 'download', 'extension', 'info', 'name', 'parent_path', 'path']


In [41]:
resultado = sih.download(arquivos, local_dir=str(pasta_raw_sih))
print(f"Tipo do resultado: {type(resultado)}")
print(f"Resultado: {resultado}")

3414473it [00:00, 1282584431.47it/s] 

Tipo do resultado: <class 'pysus.data.local.ParquetSet'>
Resultado: ../data/raw/sih/RDCE2401.parquet


In [42]:
arquivos_baixados = list(pasta_raw_sih.iterdir())
for item in arquivos_baixados:
    tipo = "pasta" if item.is_dir() else "arquivo"
    print(f"[{tipo}] {item.name}")

[pasta] RDSP2401.parquet
[pasta] RDCE2401.parquet


In [43]:
# Abrir o Parquet (funciona tanto para arquivo único quanto para diretório)
caminho_arquivo = pasta_raw_sih / "RDCE2401.parquet"

# Primeiro, ver o schema sem carregar tudo na memória
dataset = pq.ParquetDataset(str(caminho_arquivo))
schema = dataset.schema
print(f"Número de colunas: {len(schema.names)}")
print(f"\nPrimeiras 10 colunas com tipos:")
for i, nome in enumerate(schema.names[:10]):
    tipo = schema.field(nome).type
    print(f"  {i+1:3d}. {nome:20s} → {tipo}")

Número de colunas: 113

Primeiras 10 colunas com tipos:
    1. UF_ZI                → string
    2. ANO_CMPT             → string
    3. MES_CMPT             → string
    4. ESPEC                → string
    5. CGC_HOSP             → string
    6. N_AIH                → string
    7. IDENT                → string
    8. CEP                  → string
    9. MUNIC_RES            → string
   10. NASC                 → string


In [44]:
tabela = pq.read_table(str(caminho_arquivo))
df = tabela.to_pandas()
print(f"Shape: {df.shape}")  # (linhas, colunas)
print(f"Memória usada: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Shape: (46396, 113)
Memória usada: 276.0 MB


In [45]:
# Inspecionar as colunas-chave para o projeto
colunas_chave = ['DIAG_PRINC', 'MUNIC_RES', 'MORTE', 'DT_INTER', 'DT_SAIDA', 
                 'IDADE', 'SEXO', 'DIAS_PERM']

colunas_presentes = [c for c in colunas_chave if c in df.columns]
colunas_faltantes = [c for c in colunas_chave if c not in df.columns]

print(f"Colunas presentes ({len(colunas_presentes)}): {colunas_presentes}")
if colunas_faltantes:
    print(f"Colunas faltantes ({len(colunas_faltantes)}): {colunas_faltantes}")

print(f"\nPrimeiras 5 linhas das colunas-chave:")
print(df[colunas_presentes].head())

Colunas presentes (8): ['DIAG_PRINC', 'MUNIC_RES', 'MORTE', 'DT_INTER', 'DT_SAIDA', 'IDADE', 'SEXO', 'DIAS_PERM']

Primeiras 5 linhas das colunas-chave:
  DIAG_PRINC MUNIC_RES MORTE  DT_INTER  DT_SAIDA IDADE SEXO DIAS_PERM
0       T96     230440     0  20231222  20240122    73    3        31
1       M478    231360     0  20231223  20240124    14    3        32
2       K810    230440     0  20231116  20231121    17    1         5
3       G589    230440     0  20240125  20240125    65    3         0
4       G589    230440     0  20240125  20240125    64    1         0


In [46]:
# filtro do DIAG_PRINC começando com prefixo DRSAI
# TRUE para linhas do DIAG_PRINC que começa com algum dos prefixos

mascara_principal = df['DIAG_PRINC'].str.startswith(tuple(prefixos_drsai), na=False)
df_drsai_principal = df[mascara_principal].copy()

print(f"Total de internações em CE-jan/2024: {len(df):,}")
print(f"Internações DRSAI por DIAG_PRINC: {len(df_drsai_principal):,}")
print(f"Percentual: {len(df_drsai_principal) / len(df) * 100:.2f}%")

Total de internações em CE-jan/2024: 46,396
Internações DRSAI por DIAG_PRINC: 818
Percentual: 1.76%


Diferença do total de internações em SP pro CE no mesmo período:

Total de internações em SP-jan/2024: 222,849\
Internações DRSAI por DIAG_PRINC: 1,030\
Percentual: 0.46%

Total de internações em CE-jan/2024: 46,396\
Internações DRSAI por DIAG_PRINC: 818\
Percentual: 1.76%

In [47]:
# distribuição por prefixo
df_drsai_principal['PREFIXO_CID'] = df_drsai_principal['DIAG_PRINC'].str[:3]

distribuicao = df_drsai_principal['PREFIXO_CID'].value_counts()
print("Distribuição de internações DRSAI por capítulo CID:\n")
for prefixo, count in distribuicao.items():
    nome = cids_drsai.get(prefixo, '(prefixo não mapeado)')
    print(f"  {prefixo}  {nome:50s} → {count:,}")

Distribuição de internações DRSAI por capítulo CID:

  A09  Diarreia e gastroenterite origem infecciosa presumível → 538
  A04  Outras infecções bacterianas intestinais           → 167
  A08  Infecções intestinais virais                       → 83
  A05  Outras intoxicações alimentares bacterianas        → 18
  B15  Hepatite A aguda                                   → 4
  B69  Cisticercose                                       → 2
  A27  Leptospirose                                       → 2
  A00  Cólera                                             → 1
  B77  Ascaridíase                                        → 1
  A02  Outras infecções por Salmonella                    → 1
  B83  Outras helmintíases                                → 1


In [48]:
# letalidade hospitalar rela

print(f"Valores únicos em MORTE: {df_drsai_principal['MORTE'].unique()}")
print(f"Tipo de dado: {df_drsai_principal['MORTE'].dtype}")

# converte morte pra int
df_drsai_principal['MORTE_INT'] = df_drsai_principal['MORTE'].astype(int)

total_internacoes = len(df_drsai_principal)
total_obitos = df_drsai_principal['MORTE_INT'].sum()
letalidade = total_obitos / total_internacoes * 100

print(f"\n--- Letalidade hospitalar DRSAI em CE-jan/2024 ---")
print(f"Total internações DRSAI: {total_internacoes:,}")
print(f"Óbitos hospitalares DRSAI: {total_obitos:,}")
print(f"Letalidade: {letalidade:.2f}%")

Valores únicos em MORTE: ['0' '1']
Tipo de dado: object

--- Letalidade hospitalar DRSAI em CE-jan/2024 ---
Total internações DRSAI: 818
Óbitos hospitalares DRSAI: 8
Letalidade: 0.98%


No entanto, letalidade hospitalar é muito maior em SP que no CE, provavelmente por causa da densidade populacional (??)

Valores únicos em MORTE: ['0' '1']\
Tipo de dado: object

--- Letalidade hospitalar DRSAI em SP-jan/2024 ---
Total internações DRSAI: 1,030\
Óbitos hospitalares DRSAI: 23\
Letalidade: 2.23%

Valores únicos em MORTE: ['0' '1']\
Tipo de dado: object

--- Letalidade hospitalar DRSAI em CE-jan/2024 ---
Total internações DRSAI: 818\
Óbitos hospitalares DRSAI: 8\
Letalidade: 0.98%

In [49]:
# letalidade por capítulo
letalidade_por_cid = df_drsai_principal.groupby('PREFIXO_CID').agg(
    internacoes=('N_AIH', 'count'),
    obitos=('MORTE_INT', 'sum')
).reset_index()

letalidade_por_cid['letalidade_pct'] = (
    letalidade_por_cid['obitos'] / letalidade_por_cid['internacoes'] * 100
).round(2)

letalidade_por_cid['nome'] = letalidade_por_cid['PREFIXO_CID'].map(cids_drsai)

# ordena por letalidade decrescente
letalidade_por_cid = letalidade_por_cid.sort_values('letalidade_pct', ascending=False)

print("Letalidade hospitalar por capítulo DRSAI em CE-jan/2024:\n")
print(letalidade_por_cid[['PREFIXO_CID', 'nome', 'internacoes', 'obitos', 'letalidade_pct']].to_string(index=False))

Letalidade hospitalar por capítulo DRSAI em CE-jan/2024:

PREFIXO_CID                                                   nome  internacoes  obitos  letalidade_pct
        A00                                                 Cólera            1       1          100.00
        A27                                           Leptospirose            2       1           50.00
        A09 Diarreia e gastroenterite origem infecciosa presumível          538       5            0.93
        A04               Outras infecções bacterianas intestinais          167       1            0.60
        A02                        Outras infecções por Salmonella            1       0            0.00
        A08                           Infecções intestinais virais           83       0            0.00
        A05            Outras intoxicações alimentares bacterianas           18       0            0.00
        B15                                       Hepatite A aguda            4       0            0.00
      

### Conclusão do teste no CE

**CE tem proporcionalmente mais DRSAI**
- 0,46% (SP) vs 1,76% (CE) — o Ceará tem 3,8 vezes mais internações DRSAI proporcionalmente. Em números absolutos, CE tem 818 internações DRSAI com um quinto da população hospitalizada de SP. Isso confirma exatamente o que esperávamos: regiões com pior cobertura de saneamento geram mais internações por essas doenças.

**A letalidade do Ceará é mais baixa**

SP: 2,23% de letalidade DRSAI
CE: 0,98% de letalidade DRSAI

O Ceará tem letalidade hospitalar DRSAI menor que São Paulo. Possível causa: viés de seleção da internação. Em São Paulo, o sistema de saúde absorve diarreias leves no nível ambulatorial — UBS, pronto-socorros, atendimento domiciliar. Só chegam à internação os casos graves o suficiente para precisar de internação real. Logo, dentro do conjunto "internados", há concentração de casos sérios → letalidade aparente alta. No Ceará, com acesso ambulatorial mais limitado em algumas regiões, casos mais leves podem acabar internados (porque o paciente não tem onde se cuidar de outra forma, ou porque hidrata melhor no hospital). Logo, o conjunto "internados" inclui mais casos leves → letalidade aparente baixa.

A letalidade hospitalar é uma mistura de
- Gravidade real dos casos
- Critério de quem é internado (que varia por região)
- Qualidade do atendimento hospitalar

### Explorando o IBGE_DATASUS

- Importante para fazer o mapeamento de código de município e macrorregião

In [50]:
# Célula 21 — Investigar o que o PySUS oferece além de SIH/SINAN
import pysus
print(f"Versão do PySUS: {pysus.__version__}")
print(f"\nMódulos disponíveis em pysus.ftp.databases:")

import pysus.ftp.databases as dbs
import pkgutil
for modulo in pkgutil.iter_modules(dbs.__path__):
    print(f"  - {modulo.name}")

Versão do PySUS: 1.0.1

Módulos disponíveis em pysus.ftp.databases:
  - ciha
  - cnes
  - ibge_datasus
  - pni
  - sia
  - sih
  - sim
  - sinan
  - sinasc


In [51]:
# Célula 22 — Investigar o módulo ibge_datasus
from pysus.ftp.databases.ibge_datasus import IBGEDATASUS

ibge = IBGEDATASUS().load()
print(f"Tipo: {type(ibge)}")
print(f"\nAtributos públicos: {[a for a in dir(ibge) if not a.startswith('_')]}")

Tipo: <class 'pysus.ftp.databases.ibge_datasus.IBGEDATASUS'>

Atributos públicos: ['async_download', 'content', 'describe', 'download', 'files', 'format', 'ftp', 'get_files', 'load', 'metadata', 'name', 'paths']


In [52]:
# Célula 25 — Atributos básicos do módulo
print(f"name: {ibge.name}")
print(f"\nmetadata: {ibge.metadata}")
print(f"\npaths: {ibge.paths}")

name: IBGE-DataSUS

metadata: {'long_name': 'Populaçao Residente, Censos, Contagens Populacionais e Projeçoes Intercensitarias', 'source': 'ftp://ftp.datasus.gov.br/dissemin/publicos/IBGE', 'description': 'São aqui apresentados informações sobre a população residente, estratificadas por município, faixas etárias e sexo, obtidas a partir dos Censos Demográficos, Contagens Populacionais e Projeções Intercensitárias.'}

paths: (/dissemin/publicos/IBGE/POP, /dissemin/publicos/IBGE/censo, /dissemin/publicos/IBGE/POPTCU, /dissemin/publicos/IBGE/projpop)


In [53]:
# Célula 26 — Listar TODOS os arquivos disponíveis no IBGE_DATASUS
help(ibge.get_files)  # primeiro, vê a assinatura

Help on method get_files in module pysus.ftp.databases.ibge_datasus:

get_files(
    source: Literal['POP', 'censo', 'POPTCU', 'projpop'] = 'POPTCU',
    year: Union[list, str, int, NoneType] = None,
    *args,
    **kwargs
) -> List[pysus.ftp.File] method of pysus.ftp.databases.ibge_datasus.IBGEDATASUS instance
    Filters the list of `File`s according to each database file
    pattern, as UFs, Groups, Years, Months, etc. This method will
    also be responsible to look for wrong values within the file
    pattern and possible extra characters in its basename



In [54]:
# Célula 27 — Tentar listar todos os arquivos
try:
    todos_arquivos = ibge.get_files()
    print(f"Quantidade total de arquivos: {len(todos_arquivos)}")
    print(f"\nPrimeiros 20 arquivos:")
    for arq in todos_arquivos[:20]:
        print(f"  - {arq}")
except Exception as e:
    print(f"Erro: {e}")
    print("\nTentando ver o atributo 'files' direto...")
    print(f"files: {ibge.files[:20] if hasattr(ibge, 'files') else 'não tem'}")

Quantidade total de arquivos: 33

Primeiros 20 arquivos:
  - POPTBR00.zip
  - POPTBR01.zip
  - POPTBR02.zip
  - POPTBR03.zip
  - POPTBR04.zip
  - POPTBR05.zip
  - POPTBR06.zip
  - POPTBR07.zip
  - POPTBR08.zip
  - POPTBR09.zip
  - POPTBR10.zip
  - POPTBR11.zip
  - POPTBR12.zip
  - POPTBR13.zip
  - POPTBR14.zip
  - POPTBR15.zip
  - POPTBR16.zip
  - POPTBR17.zip
  - POPTBR18.zip
  - POPTBR19.zip


In [55]:
# Célula 28 — Investigar o atributo files
print(f"Tipo de ibge.files: {type(ibge.files)}")
print(f"Quantidade: {len(ibge.files) if hasattr(ibge.files, '__len__') else 'sem len'}")

# Mostrar amostra
print(f"\nPrimeiros 30 arquivos:")
for i, arq in enumerate(ibge.files[:30]):
    print(f"  {i+1}. {arq}")

Tipo de ibge.files: <class 'list'>
Quantidade: 152

Primeiros 30 arquivos:
  1. ALFBR00.zip
  2. ALFBR10.zip
  3. ALFBR91.zip
  4. ESCABR00.zip
  5. ESCABR10.zip
  6. ESCABR91.zip
  7. ESCBBR00.zip
  8. ESCBBR10.zip
  9. ESCBBR91.zip
  10. IDOSOBR00.zip
  11. IDOSOBR10.zip
  12. IDOSOBR91.zip
  13. POPBR00.zip
  14. POPBR01.zip
  15. POPBR02.zip
  16. POPBR03.zip
  17. POPBR04.zip
  18. POPBR05.zip
  19. POPBR06.zip
  20. POPBR07.zip
  21. POPBR08.zip
  22. POPBR09.zip
  23. POPBR10.zip
  24. POPBR11.zip
  25. POPBR12.zip
  26. POPBR80.zip
  27. POPBR81.zip
  28. POPBR82.zip
  29. POPBR83.zip
  30. POPBR84.zip


Concluiu-se que a base IBGE_DATASUS só tem informações relativas à população e não informações sobre o território, é preciso utilizar bases externas mesmo.

In [56]:
# Célula 30 — Verificar se requests está disponível
import requests
print(f"requests versão: {requests.__version__}")

requests versão: 2.33.1


Após algumas pesquisas (n lembro onde achei, mas fucei o datasus e achei), encontrei um mapeamento das macrorregiões de saúde com o código das macrorregiões e o código dos municípios: a Base Nacional de Macrorregiões de Saúde mantida pelo Departamento de Monitoramento, Avaliação e Disseminação de Informações Estratégicas em Saúde (DEMAS) do Ministério da Saúde, disponibilizada via plataforma LocalizaSUS/SEIDIGI a partir de 2024. Esta é a versão mais atual da regionalização do SUS.

Última atualização do painel em 26/04/2026 às 07:00:23, com dados do Departamento de Gestão Interfederativa e Participativa (DGIP/SE/MS)


In [57]:
# Abrir e ver as primeiras linhas
import pandas as pd
from pathlib import Path

caminho_macro = Path("../data/raw/ibge/macrorregioes_saude_ms_seidigi_2024.csv")

# Vamos começar simples: ler com configurações padrão e ver o que sai
df_macro = pd.read_csv(caminho_macro)

print(f"Shape: {df_macro.shape}")
print(f"\nColunas: {list(df_macro.columns)}")
print(f"\nPrimeiras 5 linhas:")
print(df_macro.head())

Shape: (5571, 11)

Colunas: ['Regiao do Pais', 'UF', 'Codigo Macrorregiao Interestadual de Saude', 'Macrorregiao Interestadual de Saude', 'Codigo Macrorregiao de Saude', 'Macrorregiao de Saude', 'Codigo Regiao de Saude', 'Regiao de Saude', 'Codigo Municipio', 'Municipio', 'Populacao Estimada IBGE 2022']

Primeiras 5 linhas:
  Regiao do Pais                UF Codigo Macrorregiao Interestadual de Saude  \
0   Centro-Oeste  Distrito Federal                                          -   
1   Centro-Oeste             Goias                                          -   
2   Centro-Oeste             Goias                                          -   
3   Centro-Oeste             Goias                                          -   
4   Centro-Oeste             Goias                                          -   

  Macrorregiao Interestadual de Saude  Codigo Macrorregiao de Saude  \
0                                   -                          5302   
1                                   -        

In [58]:
# Tipos de dados e valores nulos
print("Tipos de cada coluna:")
print(df_macro.dtypes)
print(f"\nValores nulos por coluna:")
print(df_macro.isna().sum())
print(f"\nTotal de linhas: {len(df_macro)}")

Tipos de cada coluna:
Regiao do Pais                                object
UF                                            object
Codigo Macrorregiao Interestadual de Saude    object
Macrorregiao Interestadual de Saude           object
Codigo Macrorregiao de Saude                   int64
Macrorregiao de Saude                         object
Codigo Regiao de Saude                         int64
Regiao de Saude                               object
Codigo Municipio                               int64
Municipio                                     object
Populacao Estimada IBGE 2022                  object
dtype: object

Valores nulos por coluna:
Regiao do Pais                                0
UF                                            0
Codigo Macrorregiao Interestadual de Saude    0
Macrorregiao Interestadual de Saude           0
Codigo Macrorregiao de Saude                  0
Macrorregiao de Saude                         0
Codigo Regiao de Saude                        0
Regiao de Saude   

In [59]:
# Verificar formato dos códigos de município
print("Colunas que parecem ter 'codigo' ou 'municipio' no nome:")
for col in df_macro.columns:
    if 'codigo' in col.lower() or 'municipio' in col.lower():
        print(f"  - '{col}'")

col_municipio = 'Codigo Municipio'  

if col_municipio in df_macro.columns:
    print(f"\nAmostra de valores em '{col_municipio}':")
    print(df_macro[col_municipio].head(10).tolist())
    
    print(f"\nTipo: {df_macro[col_municipio].dtype}")
    
    # Quantos dígitos tem?
    amostra = df_macro[col_municipio].astype(str).str.len()
    print(f"\nDistribuição de comprimento dos códigos:")
    print(amostra.value_counts())

Colunas que parecem ter 'codigo' ou 'municipio' no nome:
  - 'Codigo Macrorregiao Interestadual de Saude'
  - 'Codigo Macrorregiao de Saude'
  - 'Codigo Regiao de Saude'
  - 'Codigo Municipio'
  - 'Municipio'

Amostra de valores em 'Codigo Municipio':
[530010, 521880, 521190, 521310, 521850, 521930, 520013, 522040, 520440, 520430]

Tipo: int64

Distribuição de comprimento dos códigos:
Codigo Municipio
6    5571
Name: count, dtype: int64


In [60]:
# Célula 37 — Investigar a coluna de população
print("Amostra de valores na coluna 'Populacao Estimada IBGE 2022':")
print(df_macro['Populacao Estimada IBGE 2022'].head(10).tolist())
print(f"\nTipos únicos de caracteres não-dígito presentes:")
import re
amostra = df_macro['Populacao Estimada IBGE 2022'].head(50).tolist()
caracteres_estranhos = set()
for valor in amostra:
    if isinstance(valor, str):
        for char in valor:
            if not char.isdigit():
                caracteres_estranhos.add(char)
print(f"  {caracteres_estranhos}")

Amostra de valores na coluna 'Populacao Estimada IBGE 2022':
['2.817.381', '225.696', '105.729', '70.081', '48.447', '38.492', '21.568', '17.020', '16.513', '13.774']

Tipos únicos de caracteres não-dígito presentes:
  {'.'}


In [61]:
# Renomear colunas para snake_case
mapa_colunas = {
    'Regiao do Pais': 'regiao_pais',
    'UF': 'uf_nome',
    'Codigo Macrorregiao Interestadual de Saude': 'cod_mis',
    'Macrorregiao Interestadual de Saude': 'nome_mis',
    'Codigo Macrorregiao de Saude': 'cod_macrorregiao',
    'Macrorregiao de Saude': 'nome_macrorregiao',
    'Codigo Regiao de Saude': 'cod_regiao_saude',
    'Regiao de Saude': 'nome_regiao_saude',
    'Codigo Municipio': 'cod_municipio',
    'Municipio': 'nome_municipio',
    'Populacao Estimada IBGE 2022': 'populacao_2022',
}

df_macro = df_macro.rename(columns=mapa_colunas)
print("Novas colunas:")
print(list(df_macro.columns))

Novas colunas:
['regiao_pais', 'uf_nome', 'cod_mis', 'nome_mis', 'cod_macrorregiao', 'nome_macrorregiao', 'cod_regiao_saude', 'nome_regiao_saude', 'cod_municipio', 'nome_municipio', 'populacao_2022']


In [62]:
# Converter população de string com separador para inteiro
df_macro['populacao_2022'] = (
    df_macro['populacao_2022']
    .str.replace('.', '', regex=False)  # remove o separador de milhar
    .astype(int)                         # converte para inteiro
)

print(f"Tipo após conversão: {df_macro['populacao_2022'].dtype}")
print(f"\nEstatísticas básicas da população municipal:")
print(df_macro['populacao_2022'].describe().round(0))

Tipo após conversão: int64

Estatísticas básicas da população municipal:
count        5571.0
mean        36453.0
std        206501.0
min             0.0
25%          5228.0
50%         11066.0
75%         24474.0
max      11451999.0
Name: populacao_2022, dtype: float64


In [63]:
# Sanity checks da estrutura territorial
print(f"Total de municípios: {len(df_macro)}")
print(f"Total de macrorregiões de saúde únicas: {df_macro['cod_macrorregiao'].nunique()}")
print(f"Total de regiões de saúde únicas: {df_macro['cod_regiao_saude'].nunique()}")
print(f"Total de UFs: {df_macro['uf_nome'].nunique()}")

print(f"\nTop 5 macrorregiões por número de municípios:")
print(df_macro['nome_macrorregiao'].value_counts().head())

print(f"\nMunicípios por UF:")
print(df_macro['uf_nome'].value_counts().sort_index())

Total de municípios: 5571
Total de macrorregiões de saúde únicas: 121
Total de regiões de saúde únicas: 439
Total de UFs: 27

Top 5 macrorregiões por número de municípios:
nome_macrorregiao
NORTE                 233
MACRORREGIAO I        196
METROPOLITANA         185
MACRORREGIAO NORTE    149
SUL                   148
Name: count, dtype: int64

Municípios por UF:
uf_nome
Acre                    22
Alagoas                102
Amapa                   16
Amazonas                62
Bahia                  417
Ceara                  184
Distrito Federal         1
Espirito Santo          78
Goias                  246
Maranhao               217
Mato Grosso            142
Mato Grosso do Sul      79
Minas Gerais           853
Para                   144
Paraiba                223
Parana                 399
Pernambuco             185
Piaui                  224
Rio Grande do Norte    167
Rio Grande do Sul      497
Rio de Janeiro          92
Rondonia                52
Roraima                 15
Santa

In [66]:
# Validar chave com o SIH-SUS
# Usa o df_drsai_principal de SP-jan/2024 que já está em memória

# Pega os códigos únicos de municípios de residência das internações DRSAI de SP
municipios_sih = df_drsai_principal['MUNIC_RES'].astype(str).unique()
print(f"Municípios únicos no SIH (CE-jan/2024 DRSAI): {len(municipios_sih)}")
print(f"Amostra: {municipios_sih[:10]}")

# Pega os códigos da tabela do MS (filtrando só SP)
municipios_macro_sp = df_macro[df_macro['uf_nome'] == 'Ceara']['cod_municipio'].astype(str).unique()
print(f"\nMunicípios na tabela do MS (CE): {len(municipios_macro_sp)}")
print(f"Amostra: {municipios_macro_sp[:10]}")

# Quantos do SIH estão na tabela?
sih_set = set(municipios_sih)
macro_set = set(municipios_macro_sp)

ok = sih_set & macro_set            # interseção: estão nos dois
faltantes = sih_set - macro_set     # estão no SIH mas não na tabela

print(f"\nMunicípios do SIH-CE que ESTÃO na tabela: {len(ok)}")
print(f"Municípios do SIH-CE que NÃO estão na tabela: {len(faltantes)}")
if faltantes:
    print(f"Faltantes (até 10): {list(faltantes)[:10]}")

Municípios únicos no SIH (CE-jan/2024 DRSAI): 142
Amostra: ['230370' '230100' '230440' '230428' '230765' '230945' '230450' '230460'
 '230470' '230480']

Municípios na tabela do MS (CE): 184
Amostra: ['230110' '231180' '230870' '230760' '230690' '230700' '231310' '230535'
 '231150' '230445']

Municípios do SIH-CE que ESTÃO na tabela: 136
Municípios do SIH-CE que NÃO estão na tabela: 6
Faltantes (até 10): ['261220', '330010', '211130', '355030', '420540', '261350']


In [67]:
# Detecta automaticamente qual UF predomina nos dados
uf_predominante_codigo = df_drsai_principal['MUNIC_RES'].astype(str).str[:2].mode()[0]
print(f"UF predominante (código): {uf_predominante_codigo}")

# Mapa simplificado para confirmar
mapa_uf = {'11': 'Rondonia', '12': 'Acre', '13': 'Amazonas', '14': 'Roraima',
           '15': 'Para', '16': 'Amapa', '17': 'Tocantins',
           '21': 'Maranhao', '22': 'Piaui', '23': 'Ceara', '24': 'Rio Grande do Norte',
           '25': 'Paraiba', '26': 'Pernambuco', '27': 'Alagoas', '28': 'Sergipe', '29': 'Bahia',
           '31': 'Minas Gerais', '32': 'Espirito Santo', '33': 'Rio de Janeiro', '35': 'Sao Paulo',
           '41': 'Parana', '42': 'Santa Catarina', '43': 'Rio Grande do Sul',
           '50': 'Mato Grosso do Sul', '51': 'Mato Grosso', '52': 'Goias', '53': 'Distrito Federal'}

uf_nome_real = mapa_uf.get(uf_predominante_codigo, '?')
print(f"UF nome: {uf_nome_real}")

# Refazer a comparação com a UF correta
municipios_sih = set(df_drsai_principal['MUNIC_RES'].astype(str).unique())
municipios_macro_uf = set(
    df_macro[df_macro['uf_nome'] == uf_nome_real]['cod_municipio'].astype(str).unique()
)

ok = municipios_sih & municipios_macro_uf
faltantes = municipios_sih - municipios_macro_uf

print(f"\nMunicípios do SIH em {uf_nome_real}: {len(municipios_sih)}")
print(f"Municípios da tabela em {uf_nome_real}: {len(municipios_macro_uf)}")
print(f"Match: {len(ok)} de {len(municipios_sih)} ({len(ok)/len(municipios_sih)*100:.1f}%)")
print(f"Faltantes: {len(faltantes)}")
if faltantes:
    print(f"Códigos faltantes: {sorted(faltantes)[:20]}")

UF predominante (código): 23
UF nome: Ceara

Municípios do SIH em Ceara: 142
Municípios da tabela em Ceara: 184
Match: 136 de 142 (95.8%)
Faltantes: 6
Códigos faltantes: ['211130', '261220', '261350', '330010', '355030', '420540']


Os códigos faltantes correspondem a UFs: Maranhão, Pernambuco, RJ, SP e SC. São internações de pacientes que moram em outros estados. 

- Confirma a importância de usar MUNIC_RES em vez de MUNIC_MOV

In [68]:
# Célula 45 — Match contra a tabela INTEIRA (todas UFs)
municipios_sih = set(df_drsai_principal['MUNIC_RES'].astype(str).unique())
municipios_macro_total = set(df_macro['cod_municipio'].astype(str).unique())

ok = municipios_sih & municipios_macro_total
faltantes_real = municipios_sih - municipios_macro_total

print(f"Municípios distintos no SIH (CE-jan/2024 DRSAI): {len(municipios_sih)}")
print(f"Match contra a tabela inteira do MS: {len(ok)} ({len(ok)/len(municipios_sih)*100:.1f}%)")
print(f"\nFaltantes reais (não estão na tabela do MS): {len(faltantes_real)}")
if faltantes_real:
    print(f"Códigos: {sorted(faltantes_real)}")

Municípios distintos no SIH (CE-jan/2024 DRSAI): 142
Match contra a tabela inteira do MS: 142 (100.0%)

Faltantes reais (não estão na tabela do MS): 0


In [69]:
# Merge entre internações DRSAI e tabela de macrorregiões
# Garantir que os tipos batem antes do merge
df_drsai_principal['MUNIC_RES'] = df_drsai_principal['MUNIC_RES'].astype(str)
df_macro['cod_municipio'] = df_macro['cod_municipio'].astype(str)

# Merge: cada internação ganha cod_macrorregiao e nome_macrorregiao
df_drsai_com_macro = df_drsai_principal.merge(
    df_macro[['cod_municipio', 'cod_macrorregiao', 'nome_macrorregiao', 
              'uf_nome', 'populacao_2022']],
    left_on='MUNIC_RES',
    right_on='cod_municipio',
    how='left',  # mantém todas as internações, mesmo se não houver match
    indicator=True  # cria coluna _merge para diagnóstico
)

# Verificar se algum match foi perdido
print(f"Internações DRSAI antes do merge: {len(df_drsai_principal)}")
print(f"Internações DRSAI após merge: {len(df_drsai_com_macro)}")
print(f"\nResultado do merge:")
print(df_drsai_com_macro['_merge'].value_counts())

Internações DRSAI antes do merge: 818
Internações DRSAI após merge: 818

Resultado do merge:
_merge
both          818
left_only       0
right_only      0
Name: count, dtype: int64


In [70]:
# Tabela analítica protótipo: DRSAI agregada por macrorregião
df_drsai_com_macro['MORTE_INT'] = df_drsai_com_macro['MORTE'].astype(int)

# Agregação que vai virar o "esqueleto" do dataset analítico final
agregado_macro = df_drsai_com_macro.groupby(
    ['cod_macrorregiao', 'nome_macrorregiao']
).agg(
    internacoes=('MORTE_INT', 'count'),
    obitos=('MORTE_INT', 'sum'),
    populacao=('populacao_2022', 'first')  # populacao é constante por macrorregião? VEREMOS
).reset_index()

agregado_macro['letalidade_pct'] = (
    agregado_macro['obitos'] / agregado_macro['internacoes'] * 100
).round(2)

agregado_macro = agregado_macro.sort_values('internacoes', ascending=False)
print("Internações DRSAI por macrorregião — CE jan/2024:\n")
print(agregado_macro.to_string(index=False))

Internações DRSAI por macrorregião — CE jan/2024:

 cod_macrorregiao             nome_macrorregiao  internacoes  obitos  populacao  letalidade_pct
             2309                        SOBRAL          207       2      15657            0.97
             2310                     FORTALEZA          200       2     355679            1.00
             2308                        CARIRI          185       1       4841            0.54
             2307                SERTAO CENTRAL          139       3      11956            2.16
             2306       LITORAL LESTE JAGUARIBE           77       0      14001            0.00
             2606                        SERTAO            4       0      34843            0.00
             3521                         RRAS6            2       0   11451999            0.00
             2110            MACRORREGIAO NORTE            1       0    1037775            0.00
             2605 VALE DO S.FRANCISCO E ARARIPE            1       0      62372      

In [ ]:
# Forma correta de agregar população por macrorregião
populacao_macro = df_macro.groupby('cod_macrorregiao')['populacao_2022'].sum().reset_index()
populacao_macro.columns = ['cod_macrorregiao', 'populacao_macro_2022']

# Junta a populacao agregada à tabela de internações por macrorregião
agregado_macro = agregado_macro.drop(columns=['populacao'])
agregado_macro = agregado_macro.merge(populacao_macro, on='cod_macrorregiao')

# Calcula taxa por 100 mil habitantes
# A taxa é a melhor forma de comparar regiões com tamanhos diferentes, caso contrário todas as macrorregiões grandes iriam liderar
agregado_macro['taxa_internacao_100k'] = (
    agregado_macro['internacoes'] / agregado_macro['populacao_macro_2022'] * 100000
).round(2)

# Reordena colunas para apresentação
agregado_macro = agregado_macro[
    ['cod_macrorregiao', 'nome_macrorregiao', 'populacao_macro_2022', 
     'internacoes', 'taxa_internacao_100k', 'obitos', 'letalidade_pct'] 
].sort_values('taxa_internacao_100k', ascending=False)

print("Tabela analítica protótipo — DRSAI por macrorregião do CE (jan/2024):\n")
print(agregado_macro.to_string(index=False))

# Internações de pacientes residentes fora da UF do hospital são mantidas no dataset, 
# atribuídas à macrorregião de residência declarada. Esta opção preserva a validade epidemiológica do indicador 
# (a doença reflete o saneamento do local de residência), embora introduza casos esparsos em macrorregiões distantes.

Tabela analítica protótipo — DRSAI por macrorregião do CE (jan/2024):

 cod_macrorregiao             nome_macrorregiao  populacao_macro_2022  internacoes  taxa_internacao_100k  obitos  letalidade_pct
             2307                SERTAO CENTRAL                618818          139                 22.46       3            2.16
             2306       LITORAL LESTE JAGUARIBE                530927           77                 14.50       0            0.00
             2308                        CARIRI               1447729          185                 12.78       1            0.54
             2309                        SOBRAL               1644010          207                 12.59       2            0.97
             2310                     FORTALEZA               4553473          200                  4.39       2            1.00
             2606                        SERTAO                837784            4                  0.48       0            0.00
             2605 VALE DO 

Análise piloto: macrorregiões cearenses em janeiro/2024 — variabilidade de incidência (4,4 a 22,5 por 100k) e letalidade (0% a 2,2%) confirma a heterogeneidade que motiva a abordagem preditiva. A macrorregião com pior perfil duplo (incidência alta + letalidade alta) foi o Sertão Central, alinhada com indicadores de saneamento abaixo da média estadual.

Deduz-se que o Sertão Central tem cobertura inferior à da Região Metropolitana de Fortaleza. 

## Conclusão da Fase 2 — Entendimento dos Dados

### Validações realizadas
- SIH-SUS acessível via PySUS, formato Parquet, 113 colunas mapeadas
- Tabela de macrorregiões (MS/SEIDIGI 2024) compatível com SIH-SUS: 100% match
- Estrutura de internação DRSAI: 0,46% em SP, 1,76% em CE — variabilidade regional confirmada
- Letalidade hospitalar: 2,23% em SP, 0,98% em CE — sugere viés de seleção da internação

### Decisões consolidadas
- Categorização DRSAI em 3 grupos por mecanismo de transmissão
- Pipeline merge: SIH × tabela MS → cada internação ganha macrorregião
- População agregada por macrorregião: soma das populações municipais

### Próximos passos (Fase 3 — notebook 02_etl_sih.ipynb)
1. Download em massa: SIH-SUS 2017-2019 + 2023-2024, todas as UFs
2. Aplicação dos filtros DRSAI e categorização
3. Merge com tabela de macrorregiões
4. Agregação por macrorregião × trimestre
5. Salvamento em data/processed/